# 06 Cron and Webhook Automations (OpenClaw, 2026)

## What This Lesson Is
Implement event-driven and scheduled automation patterns using OpenClaw cron jobs and webhook triggers.

## Scientific Lens
- Concept: Reliable automation depends on explicit delivery mode, retry semantics, and trigger authentication.
- Measure: Automation success rate with bounded retries and verified trigger auth.
- Validity Limit: Scheduler robustness still depends on gateway uptime and external endpoint health.


## How It Works
1. Model cron job execution semantics (one-shot vs recurring, delivery modes).
2. Design webhook-triggered automation with strict token auth and route validation.
3. Run live cron/webhook commands to inspect readiness and execution state.


In [ ]:
import os
import shutil

HAS_OPENCLAW = shutil.which("openclaw") is not None
print("openclaw available:", HAS_OPENCLAW)
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("OLLAMA_BASE_URL:", os.getenv("OLLAMA_BASE_URL", "<unset>"))


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: uses real OpenClaw integration commands and skips gracefully if prerequisites are missing.


In [ ]:
# Deterministic Demo
jobs = [
    {"id":"j1","type":"one_shot","retries":0,"delivery":"none"},
    {"id":"j2","type":"recurring","retries":2,"delivery":"announce"},
    {"id":"j3","type":"recurring","retries":4,"delivery":"webhook"},
]

def effective_backoff(job):
    if job["type"] == "one_shot":
        return []
    schedule = [30, 60, 300, 900, 3600]
    return schedule[: min(job["retries"], len(schedule))]

backoffs = {j["id"]: effective_backoff(j) for j in jobs}
print(backoffs)
assert backoffs["j1"] == []
assert backoffs["j3"][-1] == 900


In [ ]:
# Live Demo
import shutil, subprocess

if not HAS_OPENCLAW:
    print("Skipping live automation demo: openclaw CLI not installed.")
else:
    cmds = [
        ["openclaw", "cron", "list"],
        ["openclaw", "cron", "status"],
        ["openclaw", "config", "get", "hooks"],
    ]
    for cmd in cmds:
        print("$", " ".join(cmd))
        p = subprocess.run(cmd, capture_output=True, text=True)
        print((p.stdout or p.stderr).strip()[:1400])


## Applied Labs
1. Design a one-shot job that escalates unresolved incidents to Telegram with `--announce`.
2. Create a webhook contract test that rejects missing/invalid bearer tokens.
3. Model cron delivery strategy differences for `announce` vs `webhook` vs `no-deliver`.

## Validation Checklist
- Backoff behavior is deterministic and aligned with documented scheduler semantics.
- Webhook auth expectations are explicit and testable.
- Live command outputs include cron and hooks operational state.

## Further Reading
- OpenClaw cron docs: https://docs.openclaw.ai/automation/cron-jobs
- OpenClaw cron CLI: https://docs.openclaw.ai/cli/cron
- OpenClaw webhooks: https://docs.openclaw.ai/automation/webhook
